# M1 a M3 - Ingestao, indexacao e retrieval

Este notebook cobre leitura do PDF, chunking, indexacao no Chroma e uma consulta simples via `RetrieverService`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from collections import Counter

from src.config import settings
from src.indexing import build_embeddings, open_chroma, rebuild_chroma
from src.ingestion import chunk_pages, load_pdf_pages
from src.retrieval import RetrieverService

In [3]:
pages = load_pdf_pages(settings.pdf_path)
print(f"PDF: {settings.pdf_path}")
print(f"Paginas lidas: {len(pages)}")
print(f"Paginas com texto: {sum(1 for page in pages if page['text'])}")
print(f"Primeira pagina (chars): {len(pages[0]['text'])}")
print(pages[0]['text'][:500])

PDF: /home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/notebooks/pablo/final_project/data/pdf/sindilojas_2025_2026.pdf
Paginas lidas: 38
Paginas com texto: 38
Primeira pagina (chars): 2878
Sindicato dos Comerciários de São Paulo Rua Formosa, 99 Centro CEP 01049-000 - São Paulo - SP Fone:. 2121-5900 e-mail: atendimento@comerciarios.org.br Sindicato do Comércio Varejista e Lojista do Comércio de São Paulo-Sindilojas-SP Rua. Cel. Xavier de Toledo, 99 - Centro Histórico de São P aulo CEP, 01048-100 - São Paulo – SP Fone: 11 2858-8400 e-mail: sindilojas@sindilojas-sp.org.br 1 de 33 SINDICATO DOS COMERCIÁRIOS DE SÃO PAULO CONVENÇÃO COLETIVA DE TRABALHO 2025-2026 Por este instrumento, o 


In [4]:
chunks = chunk_pages(
    pages,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    source=settings.pdf_path.name,
)

print(f"Chunks gerados: {len(chunks)}")
print(f"Chunk de exemplo: {chunks[0].metadata}")
print(chunks[0].page_content[:500])

Chunks gerados: 116
Chunk de exemplo: {'source': 'sindilojas_2025_2026.pdf', 'page': 1, 'chunk_id': '1-1'}
Sindicato dos Comerciários de São Paulo Rua Formosa, 99 Centro CEP 01049-000 - São Paulo - SP Fone:. 2121-5900 e-mail: atendimento@comerciarios.org.br Sindicato do Comércio Varejista e Lojista do Comércio de São Paulo-Sindilojas-SP Rua. Cel. Xavier de Toledo, 99 - Centro Histórico de São P aulo CEP, 01048-100 - São Paulo – SP Fone: 11 2858-8400 e-mail: sindilojas@sindilojas-sp.org.br 1 de 33 SINDICATO DOS COMERCIÁRIOS DE SÃO PAULO CONVENÇÃO COLETIVA DE TRABALHO 2025-2026 Por este instrumento, o 


In [5]:
chunks_por_pagina = Counter(chunk.metadata['page'] for chunk in chunks)
for page, count in chunks_por_pagina.most_common(10):
    print(f"Pagina {page}: {count} chunks")

Pagina 2: 4 chunks
Pagina 6: 4 chunks
Pagina 24: 4 chunks
Pagina 26: 4 chunks
Pagina 30: 4 chunks
Pagina 36: 4 chunks
Pagina 37: 4 chunks
Pagina 1: 3 chunks
Pagina 3: 3 chunks
Pagina 4: 3 chunks


In [6]:
embeddings = build_embeddings(settings.default_embedding_model)
collection = rebuild_chroma(chunks, settings.chroma_path, embeddings)
print(f"Collection: {collection.name}")
print(f"Documentos indexados: {collection.count()}")
print(f"Persistido em: {settings.chroma_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection: sindilojas_pdf
Documentos indexados: 116
Persistido em: /home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/notebooks/pablo/final_project/data/chroma


In [7]:
reopened_collection = open_chroma(settings.chroma_path, embeddings)
result = reopened_collection.query(
    query_texts=['qual e a vigencia da convencao coletiva?'],
    n_results=3,
)

print(f"Indice reaberto com {reopened_collection.count()} documentos")
for metadata, document in zip(result['metadatas'][0], result['documents'][0]):
    print(metadata)
    print(document[:300])
    print('-' * 80)

Indice reaberto com 116 documentos
{'source': 'sindilojas_2025_2026.pdf', 'chunk_id': '24-2', 'page': 24}
praticadas pelas administradoras de cartão de crédito. 51 - ACORDOS COLETIVOS- Considerando que a convenção coletiva é instrumento de regulação do mercado de trabalho, que assegura e garante patamares mínimo s, evitando efeitos danosos às categorias profissional e empresarial, que asse gura o equilí
--------------------------------------------------------------------------------
{'chunk_id': '26-2', 'page': 26, 'source': 'sindilojas_2025_2026.pdf'}
s fixados em lei municipal, estadual ou federal que recaírem às terças, quartas ou quintas-feiras, poderão ser gozados às segundas ou sextas-f eiras, mediante acordo entre empregador e empregado. 57 – CONCILIAÇÃO PRÉVIA, MEDIAÇÃO E ARBITRAGEM PARA SOLUÇÃO DE CONFLITOS – Objetivando o fomento às solu
--------------------------------------------------------------------------------
{'page': 26, 'source': 'sindilojas_2025_2026.pdf', 'chunk_i

In [8]:
retriever = RetrieverService(
    persist_dir=settings.chroma_path,
    embeddings=embeddings,
)

retrieved_chunks = retriever.retrieve(
    'qual e a vigencia da convencao coletiva?',
    k=3,
)

for chunk in retrieved_chunks:
    print(chunk.score, chunk.page, chunk.source, chunk.chunk_id)
    print(chunk.text[:300])
    print('-' * 80)

0.8223427534103394 24 sindilojas_2025_2026.pdf 24-2
praticadas pelas administradoras de cartão de crédito. 51 - ACORDOS COLETIVOS- Considerando que a convenção coletiva é instrumento de regulação do mercado de trabalho, que assegura e garante patamares mínimo s, evitando efeitos danosos às categorias profissional e empresarial, que asse gura o equilí
--------------------------------------------------------------------------------
0.7693333029747009 26 sindilojas_2025_2026.pdf 26-2
s fixados em lei municipal, estadual ou federal que recaírem às terças, quartas ou quintas-feiras, poderão ser gozados às segundas ou sextas-f eiras, mediante acordo entre empregador e empregado. 57 – CONCILIAÇÃO PRÉVIA, MEDIAÇÃO E ARBITRAGEM PARA SOLUÇÃO DE CONFLITOS – Objetivando o fomento às solu
--------------------------------------------------------------------------------
0.7660022377967834 26 sindilojas_2025_2026.pdf 26-3
put, as decisões das demandas submetidas à Conciliação Prévia, Mediação e Arbitr